# 🕵️ Dia 07 — HTTP para investigadores: cabeçalhos, status e redirecionamentos

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 7* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-07.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

18h18. O seu `curl` sempre recebeu *"Sistema em manutenção"*. Mas três cooperados digitaram CPF e senha num formulário. Rafa: *"Quem caiu no golpe pediu como celular Android, em português. Pede de novo. Do jeito da vítima."*

No navegador da VM: `302`, `301`, `200`... e a página pula sozinha para `/pix/regularizar`, sem nenhum `Location`.

Neste notebook você vai:

1. **Subir um servidor local que imita a camuflagem do IscaKit** ("manutenção" para `curl`, formulário inerte para celular).
2. **Escrever um seguidor de redirecionamentos** que mostra cada salto, e ver o que ele **não** segue.
3. **Ensinar o seguidor a detectar `meta refresh` e JavaScript.**
4. **Ler o HAR e desenhar as duas cadeias** da Operação Isca.
5. **Jogar o quiz "qual mecanismo?"** (6 perguntas).

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Funciona sem internet: toda célula que tenta a rede tem plano B embutido.

## 1. Um kit de mentira, na sua máquina

Nada de visitar site de golpe hoje: o notebook sobe um servidor **local** (`127.0.0.1`, só a sua máquina enxerga) que se comporta como o IscaKit. Ele:

- responde `/sa7` com `302` (o "encurtador");
- responde `/` com "Sistema em manutenção" para quem parece *scanner* (`curl`, `python-urllib`, sem `pt-BR`);
- para quem parece celular em português, devolve uma página que pula por **JavaScript** para `/pix/regularizar`;
- `/pix/regularizar` mostra um formulário **inerte** (sem `action`, sem campos de senha reais).

Também serve `/meta`, a página de *meta refresh* do Passo 3 do original.

In [ ]:
import http.server, threading, socket

PAGINA_MANUTENCAO = "<html><body><h1>Cooperativa Serra Azul</h1><p>Sistema em manutenção.</p></body></html>"
PAGINA_JS = ("<html><head><script>window.location = '/pix/regularizar';</script></head>"
             "<body>Carregando...</body></html>")
PAGINA_FORM = ("<html><body><h1>Regularize sua chave Pix</h1>"
               "<p>[FORMULÁRIO INERTE DE LABORATÓRIO — sem envio, sem campos reais]</p></body></html>")
PAGINA_META = ('<html><head><meta http-equiv="refresh" content="0; url=/pix/regularizar"></head>'
               "<body>Redirecting...</body></html>")

def parece_vitima(cabecalhos):
    """A regra de camuflagem (fictícia) do kit: celular + português."""
    ua = cabecalhos.get("User-Agent", "")
    idioma = cabecalhos.get("Accept-Language", "")
    return ("Android" in ua or "iPhone" in ua) and idioma.lower().startswith("pt")

class KitDeMentira(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/sa7":                               # o "encurtador"
            return self.responder(302, "", {"Location": "/"})
        if self.path == "/":
            corpo = PAGINA_JS if parece_vitima(self.headers) else PAGINA_MANUTENCAO
            return self.responder(200, corpo, {"Set-Cookie": "isk_sess=lab123; HttpOnly"})
        if self.path == "/pix/regularizar":
            return self.responder(200, PAGINA_FORM)
        if self.path == "/meta":
            return self.responder(200, PAGINA_META)
        self.responder(404, "não encontrado")

    def responder(self, status, corpo, extras=None):
        dados = corpo.encode("utf-8")
        self.send_response(status)                            # linha de status + Date + Server
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        for k, v in (extras or {}).items():
            self.send_header(k, v)
        self.end_headers()
        self.wfile.write(dados)

    def log_message(self, *args):                             # silencia o log no console
        pass

# Porta livre qualquer, só em 127.0.0.1 (ninguém de fora alcança).
servidor = http.server.ThreadingHTTPServer(("127.0.0.1", 0), KitDeMentira)
PORTA = servidor.server_address[1]
threading.Thread(target=servidor.serve_forever, daemon=True).start()
BASE_LAB = f"http://127.0.0.1:{PORTA}"
print("🧪 Kit de mentira rodando em", BASE_LAB)

## 2. Um seguidor de redirecionamentos que mostra cada salto

O `urllib` normalmente segue os `3xx` sozinho e esconde o caminho. Desligamos isso para ver cada `Location`, como o `curl -sIL`.

In [ ]:
import urllib.request, urllib.error, urllib.parse
from datetime import datetime, timezone

class NaoSeguir(urllib.request.HTTPRedirectHandler):
    """Faz o urllib DEVOLVER a resposta 3xx em vez de segui-la escondido."""
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

ABRIDOR = urllib.request.build_opener(NaoSeguir)
UA_CURL = "curl/8.5.0"
UA_ANDROID = "Mozilla/5.0 (Linux; Android 14) Mobile"

def seguir(url, ua=UA_CURL, idioma="en", max_saltos=10, olhar_html=False):
    """Segue a cadeia salto a salto. Devolve lista de (hora, url, status, mecanismo, destino)."""
    saltos = []
    for _ in range(max_saltos):
        req = urllib.request.Request(url, headers={"User-Agent": ua, "Accept-Language": idioma})
        hora = datetime.now(timezone.utc).strftime("%H:%M:%S.%f")[:-3] + "Z"
        try:
            r = ABRIDOR.open(req, timeout=5)
            html = r.read().decode("utf-8", "replace")
            status, destino = r.status, None
        except urllib.error.HTTPError as e:              # 3xx e 4xx chegam aqui
            status, html, destino = e.code, "", e.headers.get("Location")
        if destino:                                       # redirecionamento por cabeçalho
            saltos.append((hora, url, status, "Location", destino))
            url = urllib.parse.urljoin(url, destino)
            continue
        if olhar_html:                                    # o que o curl -L NÃO faz
            achado = redirecionamento_no_html(html)
            if achado:
                mecanismo, alvo = achado
                saltos.append((hora, url, status, mecanismo, alvo))
                url = urllib.parse.urljoin(url, alvo)
                continue
        saltos.append((hora, url, status, "fim", None))
        break
    return saltos

def redirecionamento_no_html(html):
    """Placeholder: vamos escrever de verdade na seção 3."""
    return None

def mostrar(saltos, titulo):
    print(f"== {titulo}: {sum(1 for s in saltos if s[3] != 'fim')} redirecionamento(s)")
    for hora, url, status, mec, destino in saltos:
        print(f"   {hora}  {status}  {url.replace(BASE_LAB, '')}  --{mec}--> {(destino or '').replace(BASE_LAB, '')}")

mostrar(seguir(BASE_LAB + "/sa7"), "como curl")
mostrar(seguir(BASE_LAB + "/sa7", UA_ANDROID, "pt-BR"), "como celular, mas olhando só cabeçalhos")
mostrar(seguir(BASE_LAB + "/meta"), "teste do meta refresh (Passo 3)")

Repare: mesmo fingindo ser celular, o seguidor **parou num `200`**. O formulário existe, mas o caminho até ele é JavaScript. É exatamente o que o `curl -L` faz: *"uma cadeia que parece parar num `200 OK` pode continuar no navegador"*.

## 3. Ensinando o seguidor a ler o HTML

In [ ]:
import re

def redirecionamento_no_html(html):
    """Procura meta refresh e window.location no HTML. Devolve (mecanismo, destino) ou None."""
    m = re.search(r'<meta[^>]+http-equiv=["\']?refresh["\']?[^>]+content=["\'][^"\']*url=([^"\'>]+)',
                  html, re.I)                                   # <meta http-equiv="refresh" content="0; url=...">
    if m:
        return ("meta refresh", m.group(1).strip())
    m = re.search(r'(?:window\.)?location(?:\.href)?\s*=\s*["\']([^"\']+)', html)   # window.location = '...'
    if m:
        return ("JavaScript", m.group(1))
    return None

mostrar(seguir(BASE_LAB + "/sa7", UA_CURL, "en", olhar_html=True), "como curl, lendo HTML")
mostrar(seguir(BASE_LAB + "/sa7", UA_ANDROID, "pt-BR", olhar_html=True), "como celular pt-BR, lendo HTML")
mostrar(seguir(BASE_LAB + "/meta", olhar_html=True), "meta refresh, lendo HTML")

> ⚠️ Expressão regular não executa JavaScript. Kits reais ofuscam o redirecionamento (montam a URL em pedaços, usam `atob`, esperam um clique). Para esses, o caminho é o navegador de análise com HAR, ou um *sandbox* como o urlscan.io (Fase 2). O que você ganhou aqui é o reflexo: **se parou em 200, olhe o HTML**.

## 4. As duas cadeias da Operação Isca (dados do caso)

In [ ]:
# ------------------------------------------------------------
# DADOS DO CASO (Operação Isca — tudo fictício)
# Primeiro tentamos baixar do repositório no GitHub; se não houver
# internet (ou o repo estiver fora do ar), usamos a cópia embutida.
# ------------------------------------------------------------
import urllib.request   # baixa arquivos pela web, sem instalar nada

BASE = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-07/"

# Cópias embutidas: o plano B. São idênticas aos arquivos do repo.
EMBUTIDO = {}
EMBUTIDO['cadeia.json'] = '{\n "coletado_utc": "2026-10-05T21:18:44Z",\n "coletor": "VM de análise vigia-lab01, saída por VPN de pesquisa (IP de saída 198.51.100.40)",\n "visitas": [\n  {\n   "ferramenta": "curl -sIL (User-Agent padrão do curl)",\n   "saltos": [\n    {\n     "url": "http://lnk.example/sa7",\n     "ip": "192.0.2.44",\n     "status": 302,\n     "mecanismo": "Location",\n     "location": "http://serraazul-pix.example/",\n     "server": "lnk-shortener"\n    },\n    {\n     "url": "http://serraazul-pix.example/",\n     "ip": "203.0.113.77",\n     "status": 301,\n     "mecanismo": "Location",\n     "location": "https://serraazul-pix.example/",\n     "server": "nginx"\n    },\n    {\n     "url": "https://serraazul-pix.example/",\n     "ip": "203.0.113.77",\n     "status": 200,\n     "mecanismo": "fim",\n     "location": null,\n     "server": "nginx",\n     "titulo": "Cooperativa Serra Azul (Sistema em manutenção)"\n    }\n   ]\n  },\n  {\n   "ferramenta": "navegador da VM, User-Agent de Android, Accept-Language pt-BR",\n   "saltos": [\n    {\n     "url": "http://lnk.example/sa7",\n     "ip": "192.0.2.44",\n     "status": 302,\n     "mecanismo": "Location",\n     "location": "http://serraazul-pix.example/",\n     "server": "lnk-shortener"\n    },\n    {\n     "url": "http://serraazul-pix.example/",\n     "ip": "203.0.113.77",\n     "status": 301,\n     "mecanismo": "Location",\n     "location": "https://serraazul-pix.example/",\n     "server": "nginx"\n    },\n    {\n     "url": "https://serraazul-pix.example/",\n     "ip": "203.0.113.77",\n     "status": 200,\n     "mecanismo": "JavaScript (window.location)",\n     "location": "https://serraazul-pix.example/pix/regularizar",\n     "server": "nginx"\n    },\n    {\n     "url": "https://serraazul-pix.example/pix/regularizar",\n     "ip": "203.0.113.77",\n     "status": 200,\n     "mecanismo": "fim",\n     "location": null,\n     "server": "nginx",\n     "titulo": "Regularize sua chave Pix (formulário de CPF e senha)"\n    }\n   ]\n  }\n ]\n}\n'
EMBUTIDO['vigia-lab01.har'] = '{\n "log": {\n  "version": "1.2",\n  "creator": {\n   "name": "Firefox",\n   "version": "128.0"\n  },\n  "entries": [\n   {\n    "startedDateTime": "2026-10-05T21:20:02.114Z",\n    "serverIPAddress": "192.0.2.44",\n    "request": {\n     "method": "GET",\n     "url": "http://lnk.example/sa7",\n     "headers": [\n      {\n       "name": "User-Agent",\n       "value": "Mozilla/5.0 (Linux; Android 14) Mobile"\n      },\n      {\n       "name": "Accept-Language",\n       "value": "pt-BR"\n      }\n     ]\n    },\n    "response": {\n     "status": 302,\n     "headers": [\n      {\n       "name": "Location",\n       "value": "http://serraazul-pix.example/"\n      },\n      {\n       "name": "Server",\n       "value": "lnk-shortener"\n      }\n     ]\n    }\n   },\n   {\n    "startedDateTime": "2026-10-05T21:20:02.301Z",\n    "serverIPAddress": "203.0.113.77",\n    "request": {\n     "method": "GET",\n     "url": "http://serraazul-pix.example/",\n     "headers": []\n    },\n    "response": {\n     "status": 301,\n     "headers": [\n      {\n       "name": "Location",\n       "value": "https://serraazul-pix.example/"\n      },\n      {\n       "name": "Server",\n       "value": "nginx"\n      }\n     ]\n    }\n   },\n   {\n    "startedDateTime": "2026-10-05T21:20:02.688Z",\n    "serverIPAddress": "203.0.113.77",\n    "request": {\n     "method": "GET",\n     "url": "https://serraazul-pix.example/",\n     "headers": []\n    },\n    "response": {\n     "status": 200,\n     "headers": [\n      {\n       "name": "Server",\n       "value": "nginx"\n      },\n      {\n       "name": "Set-Cookie",\n       "value": "isk_sess=c41d0e77aa; Path=/; HttpOnly; Secure"\n      },\n      {\n       "name": "Date",\n       "value": "Mon, 05 Oct 2026 21:20:02 GMT"\n      }\n     ]\n    }\n   },\n   {\n    "startedDateTime": "2026-10-05T21:20:03.022Z",\n    "serverIPAddress": "203.0.113.77",\n    "request": {\n     "method": "GET",\n     "url": "https://serraazul-pix.example/pix/regularizar",\n     "headers": [\n      {\n       "name": "Referer",\n       "value": "https://serraazul-pix.example/"\n      }\n     ]\n    },\n    "response": {\n     "status": 200,\n     "headers": [\n      {\n       "name": "Server",\n       "value": "nginx"\n      },\n      {\n       "name": "Content-Type",\n       "value": "text/html; charset=utf-8"\n      }\n     ]\n    }\n   }\n  ]\n }\n}\n'

def carregar(nome):
    """Devolve o texto do arquivo: da rede se der, senão da cópia embutida."""
    try:
        with urllib.request.urlopen(BASE + nome, timeout=5) as r:   # timeout: não travar a aula
            texto = r.read().decode("utf-8")
        if texto == EMBUTIDO[nome]:                                   # confere com a cópia local
            print(f"🌐 {nome}: baixado do GitHub (idêntico à cópia embutida)")
            return texto
        print(f"⚠️ {nome}: a versão do GitHub difere da embutida; usando a embutida (é a que os hashes do notebook esperam)")
        return EMBUTIDO[nome]
    except Exception as erro:                                         # sem rede, 404, proxy...
        print(f"📦 {nome}: usando a cópia embutida ({type(erro).__name__})")
        return EMBUTIDO[nome]

print(f"{len(EMBUTIDO)} arquivo(s) do caso disponíveis:", ", ".join(EMBUTIDO))

In [ ]:
import json, hashlib

TEXTOS = {nome: carregar(nome) for nome in EMBUTIDO}
for nome, txt in TEXTOS.items():
    print(f"{hashlib.sha256(txt.encode()).hexdigest()}  {nome}")     # o HAR é sensível: custódia + acesso restrito

CADEIA = json.loads(TEXTOS["cadeia.json"])
print("\nColetado:", CADEIA["coletado_utc"], "·", CADEIA["coletor"])

linhas = ["flowchart LR"]
for i, visita in enumerate(CADEIA["visitas"], start=1):
    linhas.append(f'    subgraph v{i}["Visita {i}: {visita["ferramenta"]}"]')
    saltos = visita["saltos"]
    for j, s in enumerate(saltos):
        rotulo = f'{s["url"]}<br/>{s["status"]} · {s["ip"]}'
        if s.get("titulo"):
            rotulo += f'<br/>{s["titulo"]}'
        linhas.append(f'        v{i}s{j}["{rotulo}"]')
    for j, s in enumerate(saltos[:-1]):
        mec = s["mecanismo"]
        if mec == "Location":
            mec = f'{s["status"]}, cabeçalho Location'
        else:
            mec += "<br/>(invisível ao curl -L)"
        linhas.append(f'        v{i}s{j} -->|"{mec}"| v{i}s{j + 1}')
    linhas.append("    end")
    n = sum(1 for s in saltos if s["mecanismo"] != "fim")
    print(f"Visita {i}: {n} redirecionamento(s), página final: {saltos[-1]['url']}")
MERMAID = "\n".join(linhas)
print()
print(MERMAID)

### Do HAR: hora, IP, status, `Location`, `Referer`

In [ ]:
har = json.loads(TEXTOS["vigia-lab01.har"])
print(f"{'Hora (UTC)':26} {'IP':14} {'St':4} URL  →  Location / Referer")
for e in har["log"]["entries"]:
    h_resp = {h["name"].lower(): h["value"] for h in e["response"]["headers"]}
    h_req = {h["name"].lower(): h["value"] for h in e["request"]["headers"]}
    extra = h_resp.get("location") or (f"(Referer: {h_req['referer']})" if "referer" in h_req else "")
    print(f"{e['startedDateTime']:26} {e['serverIPAddress']:14} {e['response']['status']:<4} {e['request']['url']}  →  {extra}")
    if "set-cookie" in h_resp:
        print(f"{'':46}⚠️ Set-Cookie: {h_resp['set-cookie'].split(';')[0]} (sessão do kit: trate o HAR como sensível)")

O `Referer` do último pedido aponta para `https://serraazul-pix.example/`: o navegador saiu da página inicial para o formulário **sem** um `3xx` no meio. É a marca de um redirecionamento feito no próprio navegador.

## 5. 🎮 Quiz: qual mecanismo?

**1** cabeçalho `3xx` + `Location` · **2** `meta refresh` · **3** JavaScript · **4** não é redirecionamento.

In [ ]:
MEC = {"1": "3xx + Location", "2": "meta refresh", "3": "JavaScript", "4": "não é redirecionamento"}
QUIZ = [
    ("HTTP/1.1 302 Found\nLocation: http://serraazul-pix.example/", "1",
     "Status 3xx com Location: o curl -L segue."),
    ('<meta http-equiv="refresh" content="0; url=/pix/regularizar">', "2",
     "Está no HTML. O curl -L informa 0 redirecionamentos."),
    ("<script>window.location = '/pix/regularizar';</script>", "3",
     "Só um navegador (ou sandbox) que executa JavaScript chega ao destino."),
    ("HTTP/2 200\nstrict-transport-security: max-age=31536000", "4",
     "HSTS manda o navegador usar HTTPS no futuro; não redireciona esta resposta."),
    ("HTTP/1.1 301 Moved Permanently\nLocation: https://serraazul-pix.example/", "1",
     "301 é permanente; o navegador pode guardar em cache."),
    ("<a href='/pix/regularizar'>Clique aqui para regularizar</a>", "4",
     "Um link depende do clique da vítima. Não é redirecionamento automático (mas é a isca!)."),
]
MODO_JOGO = True

acertos = 0
print("\n".join(f"{k} = {v}" for k, v in MEC.items()))
for i, (trecho, certo, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}]\n   " + trecho.replace("\n", "\n   "))
    if MODO_JOGO:
        r = ""
        while r not in MEC:
            r = input("   Mecanismo (1-4): ").strip()
        acertos += (r == certo)
        print("   ✅ Acertou!" if r == certo else "   ❌ Não foi dessa vez.")
    print(f"   Resposta: {MEC[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido. 🥷" if acertos >= 5 else "— meta: 5 de 6.")

## 6. 📝 Artefatos: cadeias e tabela de cabeçalhos

In [ ]:
from pathlib import Path

finais = [v["saltos"][-1] for v in CADEIA["visitas"]]
cookie_curl = "isk_sess"                                          # do Dia 2 (headers.txt)
har_final = har["log"]["entries"][-2]                              # página inicial servida ao celular
h_final = {h["name"].lower(): h["value"] for h in har_final["response"]["headers"]}
tabela = f"""| Cabeçalho | Página final do curl | Página final do celular | O que ele sugere |
|---|---|---|---|
| Status | {finais[0]['status']} | {finais[1]['status']} | Os dois "deram certo"; o conteúdo é que difere |
| URL final | {finais[0]['url']} | {finais[1]['url']} | Camuflagem por User-Agent/idioma (inferência) |
| Server | {finais[0]['server']} | {finais[1]['server']} | Software genérico; não identifica o operador |
| Cabeçalhos de CDN / cache | nenhum | nenhum | Sugere servidor direto, sem CDN na frente (inferência) |
| Nomes em Set-Cookie | {cookie_curl} | {h_final.get('set-cookie', '').split('=')[0]} | Mesmo nome de cookie nas duas visitas: mesmo kit (inferência) |
| Date (relógio do servidor) vs o seu UTC | ver headers.txt do Dia 2 | {h_final.get('date')} vs {har_final['startedDateTime']} | Relógio do servidor alinhado ao UTC (diferença < 1 s) |
"""
Path("notas").mkdir(exist_ok=True)
Path("notas/day07-redirects-isca.md").write_text("```mermaid\n" + MERMAID + "\n```\n", encoding="utf-8")
Path("notas/day07-headers-isca.md").write_text(tabela, encoding="utf-8")
print(tabela)
print("Ressalva para o item E2 do Dia 2: 'conteúdo servido a cliente curl via VPN de pesquisa;",
      "a mesma URL serve formulário a celulares pt-BR (Dia 7)'.")
print("\n💾 notas/day07-redirects-isca.md · notas/day07-headers-isca.md")

servidor.shutdown()                                               # desliga o kit de mentira
print("🧪 Kit de mentira desligado.")
try:
    from google.colab import files
    files.download("notas/day07-redirects-isca.md"); files.download("notas/day07-headers-isca.md")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta notas/)")

## ✅ Antes de fechar o Dia 7

- [ ] Viu o seguidor parar num `200` e entendeu por quê
- [ ] As duas cadeias de `lnk.example/sa7` desenhadas lado a lado
- [ ] HAR com hash e marcado como sensível
- [ ] Ressalva escrita no item E2 do Dia 2
- [ ] Quiz com pelo menos 5/6
- [ ] Fez também a prática original (`http://wikipedia.org`, segunda cadeia, *meta refresh* local, DevTools)

**Amanhã (Dia 8):** o cadeado do site falso. O certificado foi emitido na véspera do disparo, e a lista de nomes nele traz uma surpresa.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>